# Noise Diagnostics and Detectability
This notebook performs noise statistics and detectability analysis for each Asym_* folder using the generated ingress, mid, and egress spectra.

In [1]:
import numpy as np
import pandas as pd
import os

In [2]:
work_dir = os.getcwd()
def load_phase_data(folder):
    base = os.path.join(work_dir, folder)
    def load_one(phase):
        path = os.path.join(base, f"{phase}.txt")
        data = np.loadtxt(path)
        return data
    return (load_one("ingress"), load_one("mid"), load_one("egress"))
def noise_diagnostics(folder):
    ing, mid, egr = load_phase_data(folder)
    for phase, data in [("Ingress", ing), ("Mid", mid), ("Egress", egr)]:
        λ = data[:, 0]
        total_flux = data[:, -1]
        noise = data[:, 2]
        snr = np.divide(total_flux, noise, out=np.zeros_like(total_flux), where=noise > 0)
        print(f"\n{folder} - {phase}")
        print(f"  Mean flux: {np.mean(total_flux):.6e}")
        print(f"  Flux min/max: {np.min(total_flux):.6e} / {np.max(total_flux):.6e}")
        print(f"  Mean noise: {np.mean(noise):.6e}")
        print(f"  Noise min/max: {np.min(noise):.6e} / {np.max(noise):.6e}")
        print(f"  Mean SNR: {np.mean(snr):.2f}, min: {np.min(snr):.2f}, max: {np.max(snr):.2f}")
        print(f"  Mean σ/F: {np.mean(noise/total_flux):.4f} ({100 * np.mean(noise/total_flux):.2f}%)")
        region_data = []
        for r1, r2 in [(0.80, 1.00), (1.00, 1.16), (1.16, 1.30), (1.30, 1.50), (1.50, 1.70), (1.70, 1.95), (1.95, 2.20)]:
            mask = (λ >= r1) & (λ <= r2)
            region_data.append({
                'Region [μm]': f"{r1}-{r2}",
                '<Flux>': np.mean(total_flux[mask]),
                '<σ>': np.mean(noise[mask]),
                '<SNR>': np.mean(snr[mask])
            })
        display(pd.DataFrame(region_data))
def detectability_analysis(folder):
    ing, egr = load_phase_data(folder)[0], load_phase_data(folder)[2]
    λ = ing[:, 0]
    f_ing = ing[:, -1]
    f_egr = egr[:, -1]
    ni = ing[:, 2]
    ne = egr[:, 2]
    σ_comb = np.sqrt(ni**2 + ne**2)
    diff_raw = f_ing - f_egr
    snr_diff = np.divide(diff_raw, σ_comb, out=np.zeros_like(diff_raw), where=σ_comb > 0)
    print(f"\n{folder} - Detectability")
    print(f"  Mean diff: {np.mean(diff_raw):.6e}")
    print(f"  Std diff: {np.std(diff_raw):.6e}")
    print(f"  Max |diff|: {np.max(np.abs(diff_raw)):.6e}")
    print(f"  Mean SNR: {np.mean(snr_diff):.4f}")
    print(f"  Min SNR: {np.min(snr_diff):.4f}, max SNR: {np.max(snr_diff):.4f}")
    region_data = []
    for r1, r2 in [(0.80, 1.00), (1.00, 1.16), (1.16, 1.30), (1.30, 1.50), (1.50, 1.70), (1.70, 1.95), (1.95, 2.20)]:
        mask = (λ >= r1) & (λ <= r2)
        region_data.append({
            'Region [μm]': f"{r1}-{r2}",
            '<SNR>': np.mean(snr_diff[mask]),
            'Max SNR': np.max(snr_diff[mask])
        })
    display(pd.DataFrame(region_data))

In [3]:
asymmetry_levels = np.arange(0.1, 1.01, 0.1)
folders = [f"Asym_{a:.1f}" for a in asymmetry_levels]
for folder in folders:
    noise_diagnostics(folder)
    detectability_analysis(folder)


Asym_0.1 - Ingress
  Mean flux: 1.705596e-02
  Flux min/max: 1.972520e-03 / 2.002780e-02
  Mean noise: 4.140634e-04
  Noise min/max: 1.846540e-04 / 5.995350e-04
  Mean SNR: 41.05, min: 10.50, max: 51.35
  Mean σ/F: 0.0262 (2.62%)


,Region [μm],<Flux>,<σ>,<SNR>
0,0.8-1.0,0.018783,0.000375,50.112187
1,1.0-1.16,0.018493,0.000388,47.609706
2,1.16-1.3,0.019251,0.000415,46.338978
3,1.3-1.5,0.013109,0.000355,35.217229
4,1.5-1.7,0.019660,0.000479,41.029800
5,1.7-1.95,0.012531,0.000404,28.998706
6,1.95-2.2,0.017158,0.000533,31.847399



Asym_0.1 - Mid
  Mean flux: 1.691403e-02
  Flux min/max: 1.960180e-03 / 1.988500e-02
  Mean noise: 4.140967e-04
  Noise min/max: 1.846650e-04 / 5.995810e-04
  Mean SNR: 40.70, min: 10.43, max: 50.84
  Mean σ/F: 0.0265 (2.65%)


,Region [μm],<Flux>,<σ>,<SNR>
0,0.8-1.0,0.018602,0.000375,49.624889
1,1.0-1.16,0.018323,0.000388,47.169712
2,1.16-1.3,0.019085,0.000416,45.933853
3,1.3-1.5,0.013019,0.000355,34.974382
4,1.5-1.7,0.019507,0.000479,40.708134
5,1.7-1.95,0.012442,0.000404,28.792672
6,1.95-2.2,0.017037,0.000533,31.621184



Asym_0.1 - Egress
  Mean flux: 1.681449e-02
  Flux min/max: 1.952510e-03 / 1.978910e-02
  Mean noise: 4.141200e-04
  Noise min/max: 1.846730e-04 / 5.996130e-04
  Mean SNR: 40.46, min: 10.39, max: 50.47
  Mean σ/F: 0.0266 (2.66%)


...

,Region [μm],<Flux>,<σ>,<SNR>
0,0.8-1.0,0.018602,0.000375,49.624889
1,1.0-1.16,0.018323,0.000388,47.169712
2,1.16-1.3,0.019085,0.000416,45.933853
3,1.3-1.5,0.013019,0.000355,34.974382
4,1.5-1.7,0.019507,0.000479,40.708134
5,1.7-1.95,0.012442,0.000404,28.792672
6,1.95-2.2,0.017037,0.000533,31.621184



Asym_1.0 - Egress
  Mean flux: 3.006287e-03
  Flux min/max: 4.183370e-04 / 4.030080e-03
  Mean noise: 4.174329e-04
  Noise min/max: 1.860550e-04 / 6.042910e-04
  Mean SNR: 7.15, min: 2.21, max: 8.82
  Mean σ/F: 0.1454 (14.54%)


,Region [μm],<Flux>,<σ>,<SNR>
0,0.8-1.0,0.002925,0.000378,7.742302
1,1.0-1.16,0.002993,0.000391,7.655464
2,1.16-1.3,0.003263,0.000419,7.793632
3,1.3-1.5,0.002607,0.000358,6.991950
4,1.5-1.7,0.003620,0.000483,7.505071
5,1.7-1.95,0.002450,0.000407,5.677162
6,1.95-2.2,0.003403,0.000537,6.294910



Asym_1.0 - Detectability
  Mean diff: 2.126516e-03
  Std diff: 7.775178e-04
  Max |diff|: 3.196020e-03
  Mean SNR: 3.6152
  Min SNR: 0.6186, max SNR: 5.9074


,Region [μm],<SNR>,Max SNR
0,0.8-1.0,5.209730,5.907409
1,1.0-1.16,4.707060,5.431657
2,1.16-1.3,4.303665,4.588622
3,1.3-1.5,2.405563,4.449180
4,1.5-1.7,3.321119,3.764834
5,1.7-1.95,2.046695,3.615357
6,1.95-2.2,2.198279,2.731562
